# Legal RAG Bench: Dataset and Evaluation

This notebook audits the full dataset and demonstrates a shared retrieval evaluator.
It covers corpus structure, passage and section lengths, query/evidence labels, the
team split, and metric calculations. Saved tables come from the pinned source revision.

**Finding:** the released data has 100 single-gold questions and no multi-gold
questions. It supports semantic retrieval experiments, but does not directly measure
retrieval of several jointly required passages. No retrieval model scores are presented.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'config.json').exists() and (ROOT / 'legal_rag_bench_a/config.json').exists():
    ROOT = ROOT / 'legal_rag_bench_a'
assert (ROOT / 'legal_rag_a').exists(), 'Open from the legal_rag_bench_a folder or its parent'
sys.path.insert(0, str(ROOT))
OUT = ROOT / 'results'
profile = json.loads((OUT / 'dataset_profile.json').read_text(encoding='utf-8'))
config = json.loads((ROOT / 'config.json').read_text(encoding='utf-8'))

## 1. Source, scope, and reproducibility

The source is the Victorian Criminal Charge Book, distributed as pre-chunked Markdown
passages. The official corpus and QA configurations both have a `test` split only.
Source files are downloaded separately and verified by SHA-256; raw data is excluded
from the GitHub upload. The summary cells run from the included aggregate files.

In [2]:
display(pd.DataFrame([
    ('Dataset', profile['dataset']), ('Pinned revision', profile['revision']),
    ('Passages', profile['n_passages']), ('Questions', profile['n_queries']),
    ('Unique gold passages', profile['n_unique_gold_passages']),
    ('ID-derived section groups', profile['n_id_derived_sections']),
    ('Text policy', config['text_policy']),
], columns=['Setting', 'Value']))

,Setting,Value
0,Dataset,isaacus/legal-rag-bench
1,Pinned revision,db0b31dc6d195ce9916897e1ac5e4e6209736c8a
2,Passages,4876
3,Questions,100
4,Unique gold passages,95
5,ID-derived section groups,724
6,Text policy,text_only


## 2. Actual schema and evidence mapping

The corpus has `id`, `title`, `text`, and nullable `footnotes`. Query IDs are integers
in the source and normalized to strings. Each `relevant_passage_id` resolves to an
original corpus ID. Reference answers are reserved for generation evaluation; neither
answers nor gold labels are part of retrieval text.

In [3]:
schema_rows = []
for subset, fields in profile['schema'].items():
    for field, details in fields.items():
        schema_rows.append({'Subset': subset, 'Field': field,
                            'Types': ', '.join(details['types']),
                            'Missing': details['missing'], 'Null': details['null']})
display(pd.DataFrame(schema_rows))
example = json.loads((OUT / 'schema_examples.json').read_text(encoding='utf-8'))
display(pd.DataFrame([example['link_example']]))

,Subset,Field,Types,Missing,Null
0,corpus,footnotes,"NoneType, str",0,2969
1,corpus,id,str,0,0
2,corpus,text,str,0,0
3,corpus,title,str,0,0
4,qa,answer,str,0,0
5,qa,id,int,0,0
6,qa,question,str,0,0
7,qa,relevant_passage_id,str,0,0


,query_id,gold_ids,resolved_passage_exists
0,1,[1.2-c2-s2],True


## 3. Lengths and corpus quality

Lengths below are **whitespace-delimited words**, not tokenizer counts. The paper
describes chunking to 512 Kanon tokens; another embedding tokenizer can produce
different counts. Section lengths sum released passage bodies grouped by ID prefix.
They do not reproduce exact source Word-document lengths.

Repeated text is retained to preserve location IDs and gold mappings. Footnotes stay
available as metadata. Indexing title/footnote content is a separate text-policy choice
that must be held constant across retrieval methods.

In [4]:
lengths = pd.DataFrame(profile['lengths']).T
display(lengths[['count', 'min', 'median', 'p95', 'max', 'mean']].round(2))
display(pd.DataFrame([
    ('Repeated text rows beyond first occurrence', profile['duplicate_text_rows']),
    ('Passages with footnotes', profile['n_passages_with_footnotes']),
], columns=['Check', 'Count']))
passage_lengths = pd.read_csv(OUT / 'passage_lengths.csv')
display(passage_lengths.sort_values('text_words', ascending=False).head(5))

,count,min,median,p95,max,mean
passage_text_words,4876.0,2.0,209.0,382.00,452.0,209.07
passage_text_characters,4876.0,10.0,1216.0,2280.25,2715.0,1234.99
id_derived_section_words,724.0,77.0,790.5,4284.35,11424.0,1408.03
question_words,100.0,4.0,46.5,91.00,162.0,49.10
answer_words,100.0,14.0,49.5,80.20,118.0,49.65


,Check,Count
0,Repeated text rows beyond first occurrence,218
1,Passages with footnotes,1907


,passage_id,section_id,text_characters,text_words,footnote_words,title_words
4735,9.1.2.3-c6-s2,9.1.2.3,2648,452,0,2
1728,7.3.2.1-c3-s1,7.3.2.1,2514,442,24,4
622,4.12-c7-s2,4.12,2636,439,250,8
2894,7.4.7.1-c2-s1,7.4.7.1,2699,438,0,3
4134,7.9.1-c2-s1,7.9.1,2543,438,0,2


## 4. Single- versus multi-evidence labels

Count annotations rather than inferring evidence requirements from question length.
Every released question has one gold passage. The benchmark intentionally stresses
semantic matching through lexical differences, but its labels do not validate
multi-passage completeness. Unannotated useful passages can still exist.

In [5]:
display(pd.DataFrame([
    {'Evidence labels': 'Single gold passage', 'Questions': profile['single_gold_queries'],
     'Percent': 100 * profile['single_gold_fraction']},
    {'Evidence labels': 'Multiple gold passages', 'Questions': profile['multi_gold_queries'],
     'Percent': 100 * profile['multi_gold_fraction']},
]))

,Evidence labels,Questions,Percent
0,Single gold passage,100,100.0
1,Multiple gold passages,0,0.0


## 5. Team development/test split

The optional team split is 20 development and 80 test questions. It groups queries
sharing a gold passage before deterministic allocation, so those groups do not cross
the split. Legal topics can still overlap. All 4,876 passages remain searchable.

Use development questions to tune parameters, freeze choices, and then evaluate
held-out questions. This is our exploratory split, not an official dataset split.
Results over all 100 questions remain available using `official_test`; after tuning
on a subset, those all-100 results are not wholly held-out.

In [6]:
splits = pd.read_csv(OUT / 'query_splits.csv', dtype={'query_id': str})
display(splits.groupby('split').size().rename('Questions').to_frame())
display(splits.head())

,Questions
split,
dev,20
test,80


,query_id,split,split_id
0,1,test,legal-rag-a-sha256-gold-group-20pct-v1
1,2,test,legal-rag-a-sha256-gold-group-20pct-v1
2,3,dev,legal-rag-a-sha256-gold-group-20pct-v1
3,4,test,legal-rag-a-sha256-gold-group-20pct-v1
4,5,test,legal-rag-a-sha256-gold-group-20pct-v1


## 6. Shared metric calculation

Precision@k = relevant results / k. Recall@k = relevant results / annotated gold count.
MRR@k is the reciprocal rank of the first gold hit; nDCG@k uses binary relevance and
logarithmic rank discounting. Every metric is computed per query and macro-averaged.

Evidence coverage equals recall. All-gold inclusion is one only when every annotated
gold ID is returned. With one gold per query, both coincide with hit rate. They do not
prove answer completeness. Precision among actual returned results is reported separately
from fixed-k precision. Duplicate IDs are rejected.

The following **synthetic calculation examples** test two required evidence passages;
they are not Legal RAG Bench annotations or model predictions.

In [7]:
from legal_rag_a import query_metrics, evaluate

rows = []
for name, ranking in [('Partial evidence', ['X', 'A', 'Y']),
                      ('All evidence', ['X', 'A', 'B']), ('Empty output', [])]:
    rows.append({'Example': name, **query_metrics(ranking, {'A', 'B'}, 3)})
display(pd.DataFrame(rows)[['Example', 'precision_at_k', 'recall_at_k',
                           'evidence_coverage_at_k', 'all_gold_included_at_k', 'hit_rate_at_k',
                           'mrr_at_k', 'ndcg_at_k']].rename(columns={
    'precision_at_k': 'Precision@3', 'recall_at_k': 'Recall@3',
    'evidence_coverage_at_k': 'Evidence Coverage@3',
    'all_gold_included_at_k': 'All Evidence Hit@3', 'hit_rate_at_k': 'Hit Rate@3',
    'mrr_at_k': 'MRR@3', 'ndcg_at_k': 'nDCG@3'}).round(4))

,Example,Precision@3,Recall@3,Evidence Coverage@3,All Evidence Hit@3,Hit Rate@3,MRR@3,nDCG@3
0,Partial evidence,0.3333,0.5,0.5,0.0,1.0,0.5,0.3869
1,All evidence,0.6667,1.0,1.0,1.0,1.0,0.5,0.6934
2,Empty output,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000


## 7. Full-dataset evaluator controls

The oracle control supplies each query's gold ID and must have recall one. The empty
control must have recall zero. These controls were executed for all 100 questions
at k = 1, 3, 5, and 10. They verify the evaluation path, not retrieval performance.
An oracle returning one relevant result has Precision@5 = 0.2 under fixed-k precision.

In [8]:
controls = pd.read_csv(OUT / 'evaluator_controls.csv')
display(controls[['control', 'k', 'n_queries', 'precision_at_k', 'recall_at_k',
                  'hit_rate_at_k', 'all_gold_included_at_k', 'evidence_coverage_at_k',
                  'mrr_at_k', 'ndcg_at_k']].rename(columns={
    'all_gold_included_at_k': 'All Evidence Hit@k',
    'evidence_coverage_at_k': 'Evidence Coverage@k'}))
assert controls.loc[controls.control.str.startswith('oracle'), 'recall_at_k'].eq(1).all()
assert controls.loc[controls.control.str.startswith('empty'), 'recall_at_k'].eq(0).all()

,control,k,n_queries,precision_at_k,recall_at_k,hit_rate_at_k,All Evidence Hit@k,Evidence Coverage@k,mrr_at_k,ndcg_at_k
0,oracle_label_check_NOT_A_RETRIEVER,1,100,1.000000,1.0,1.0,1.0,1.0,1.0,1.0
1,oracle_label_check_NOT_A_RETRIEVER,3,100,0.333333,1.0,1.0,1.0,1.0,1.0,1.0
2,oracle_label_check_NOT_A_RETRIEVER,5,100,0.200000,1.0,1.0,1.0,1.0,1.0,1.0
3,oracle_label_check_NOT_A_RETRIEVER,10,100,0.100000,1.0,1.0,1.0,1.0,1.0,1.0
4,empty_output_check_NOT_A_RETRIEVER,1,100,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
5,empty_output_check_NOT_A_RETRIEVER,3,100,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
6,empty_output_check_NOT_A_RETRIEVER,5,100,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
7,empty_output_check_NOT_A_RETRIEVER,10,100,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


## 8. Load records for retrieval or interactive inspection

Set `LOAD_SOURCE = True` to download/validate the pinned source and load canonical
records. The initial text policy indexes the passage body only. Keep passage IDs
beside vectors/BM25 rows so every method returns the original IDs. Avoid feeding
`gold_ids` or `answer` into retrieval or graph construction.

In [9]:
LOAD_SOURCE = False
if LOAD_SOURCE:
    from legal_rag_a.data import download
    from legal_rag_a import load_benchmark, retrieval_text
    download()
    corpus, questions = load_benchmark()
    passage_ids = [p['id'] for p in corpus]
    texts = [retrieval_text(p) for p in corpus]
    query_texts = {q['id']: q['question'] for q in questions}
    print(f'Loaded {len(texts)} passages and {len(query_texts)} questions.')
    # A retriever returns original passage IDs in decreasing rank order.
    # per_query, aggregate = evaluate(questions, predictions, set(passage_ids))

## 9. Live retrieval timing and the shared results table

`benchmark_retrievers` times actual single-query calls, including materialization of
returned IDs. Build models and indexes first. Each adapter accepts `(question_text, k)`
and must include query encoding, search, fusion/refinement, and evidence lookup before
returning original passage IDs. Generation and index construction are excluded.

Use three warm-ups per method and three seeded, interleaved repetitions on the same
questions. GPU adapters should supply synchronization hooks. Report the device/model,
text policy, cache behavior, and software versions with results. These are single-query
measurements, not batch-throughput estimates. First-repeat rankings feed the evaluator.

No retrieval implementation is registered by default. The cell below provides executable
integration for teammates; it produces no invented model latencies. Timer unit tests use
stubs and a controlled clock and are not performance measurements.

In [10]:
from legal_rag_a import benchmark_retrievers, load_benchmark
from legal_rag_a.__main__ import write_csv, write_json

RETRIEVERS = {}  # Example after building an index: {'BM25': bm25_retrieve}
GPU_SYNC = {}  # Example for a CUDA adapter: {'Dense': torch.cuda.synchronize}
if RETRIEVERS:
    corpus, questions = load_benchmark()
    test_ids = set(splits.loc[splits['split'] == 'test', 'query_id'])
    selected = [q for q in questions if q['id'] in test_ids]
    raw, timing, rankings = benchmark_retrievers(
        RETRIEVERS, selected, k=5, warmup=3, repeats=3, synchronize=GPU_SYNC)
    output = OUT / 'local_live_retrieval'
    write_csv(output / 'latency_raw.csv', raw)
    write_csv(output / 'latency_summary.csv', timing)
    write_json(output / 'rankings.json', rankings)
    combined = []
    for name, predictions in rankings.items():
        details, summary = evaluate(selected, predictions, {p['id'] for p in corpus}, ks=(5,))
        combined.append({'Method': name, **summary[0]})
    table = pd.DataFrame(combined).merge(
        pd.DataFrame(timing)[['method', 'mean_ms', 'p95_ms']], left_on='Method', right_on='method')
    table = table.rename(columns={
        'precision_at_k': 'Precision@5', 'recall_at_k': 'Recall@5',
        'hit_rate_at_k': 'Hit Rate@5', 'all_gold_included_at_k': 'All Evidence Hit@5',
        'evidence_coverage_at_k': 'Evidence Coverage@5', 'mrr_at_k': 'MRR@5',
        'ndcg_at_k': 'nDCG@5', 'mean_ms': 'Latency mean (ms)', 'p95_ms': 'Latency p95 (ms)'})
    table = table[['Method', 'Precision@5', 'Recall@5', 'Hit Rate@5', 'All Evidence Hit@5',
                   'Evidence Coverage@5', 'MRR@5', 'nDCG@5', 'Latency mean (ms)', 'Latency p95 (ms)']]
    write_csv(output / 'comparison.csv', table.to_dict('records'))
    display(table.round(4))
else:
    print('Timing utility ready. Register real retrievers to produce measured comparison results.')

Timing utility ready. Register real retrievers to produce measured comparison results.


## 10. Team handoff and conclusions

BM25, dense, hybrid, and graph-assisted methods should produce the same prediction
format: `{"query_id": "...", "passage_ids": ["...", "..."]}`. The CLI evaluates
exactly the requested split; details and commands are in README.md.

This dataset provides a useful semantic retrieval test, with an existing passage
corpus and stable evidence IDs. It is not a ready-made multi-evidence or full-document
benchmark. Additional splitting/merging changes evaluation meaning and requires a
separate mapping policy. A future completeness study needs annotated required facts
or jointly necessary evidence sets. Raw retrieval scores cannot replace answer
correctness or groundedness assessments.

Sources: [dataset](https://huggingface.co/datasets/isaacus/legal-rag-bench),
[paper](https://arxiv.org/html/2603.01710v1), and
[official evaluation code](https://github.com/isaacus-dev/legal-rag-bench).
See DATA_LICENSE.md for the upstream license wording discrepancy.